In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
#pip install xlrd

In [3]:
#Download the file
df = pd.read_excel("GSAF5.xls")
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
#Eliminar columnas por nombre 
columns_to_drop = { 
    'pdf',
    'href',
    'href formula',
    'Case Number',
    'Case Number.1',
    'original order',
    'Unnamed: 21',
    'Unnamed: 22'
}
df = df.drop(columns=columns_to_drop)

In [5]:
df.shape

(7065, 15)

In [6]:
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,puncture mark to left thumb,N,0540hrs,Unknown,Bob Myatt GSAF
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,Left arm torn off in the attack below the elbow,Y,1628hrs,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,Injuries to upper limbs,N,?,Unknown,Andy Currie: Province Sud:
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,Taken by shark body recovered with multiple in...,Y,1200hrs,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,Hand Injury,N,0800hrs,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...


#### ver los valores nulos y duplicados

In [7]:
df.isna().sum()/df.shape[0]

Date         0.000000
Year         0.000283
Type         0.002548
Country      0.007077
State        0.068931
Location     0.080255
Activity     0.082803
Name         0.030998
Sex          0.081953
Age          0.423921
Injury       0.004954
Fatal Y/N    0.079406
Time         0.499222
Species      0.443171
Source       0.002831
dtype: float64

In [8]:
df.duplicated().sum()

1

In [9]:
filas_duplicadas = df[df.duplicated(keep=False)]
filas_duplicadas

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
5436,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman
5437,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman


In [10]:
#eliminando duplicados 
df = df.drop(5436)

In [11]:
df.shape

(7064, 15)

### Column Type

In [12]:
#Looks Type 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Questionable', 'unprovoked',
       ' Provoked', 'Watercraft', 'Sea Disaster', nan, '?', 'Unconfirmed',
       'Unverified', 'Invalid', 'Under investigation', 'Boat'],
      dtype=object)

In [13]:
#Clean type
df['Type'] = df['Type'].astype(str).str.strip().str.capitalize()

#Unificar valores raros a 'Unkown
df['Type'] = df['Type'].replace({
    'Unverified': 'Unknown',
    'Unconfirmed': 'Unknown',
    '?': 'Unknown',
    'Nan': 'Unknown',         
    'Invalid': 'Unknown',
    'Under investigation': 'Unknown',
    'Questionable' : 'Unknown'
})

#unificar 'Unprovoked' y 'Provoked' (ya los capitalizamos)
df['Type'] = df['Type'].replace({'Unprovoked': 'Unprovoked', 'Provoked': 'Provoked'})

#otros tipos 
df['Type'] = df['Type'].replace({'Watercraft': 'Other', 'Sea disaster': 'Other', 'Boat': 'Other'})

In [14]:
#Los cambios se hicieron efectivos 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Unknown', 'Other'], dtype=object)

### Hipotesis 2: Mueren mas hombres que mujeres?

In [15]:
df['Sex'].unique()

array(['M', 'F', 'F ', 'M ', nan, ' M', 'm', 'lli', 'M x 2', 'N', '.'],
      dtype=object)

#### Eliminacion de Nulos de ['Sex']

In [16]:
#Revisando los nulos de la columna 'Sex'
df['Sex'].isnull().sum()

579

In [17]:
#Quitamos espacios en blanco y ponemos todos en mayuscula, para unificar valores
df['Sex'] = df['Sex'].str.strip().str.upper()
df['Sex'].unique()


array(['M', 'F', nan, 'LLI', 'M X 2', 'N', '.'], dtype=object)

In [18]:
#Englobamos valores raros a 'Unkown'
df['Sex'] = df['Sex'].replace({
    'M X 2': 'Unknown',
    'LLI': 'Unknown',
    'N': 'Unknown',
    '.': 'Unknown'       
})

df['Sex'].unique()

array(['M', 'F', nan, 'Unknown'], dtype=object)

In [19]:
df['Sex'].isna().sum()

579

In [20]:
#Eliminacion de 'nan' de nuestra columna Sex
df = df.dropna(subset=['Sex'])

In [21]:
#Verificamos que se eliminaron los 'nan'
df['Sex'].unique()

array(['M', 'F', 'Unknown'], dtype=object)

In [22]:
#Utilizamos describe para ver cuando es el valor mas frecuente en Sex que es hombre
df['Sex'].describe()

count     6485
unique       3
top          M
freq      5670
Name: Sex, dtype: object

#### Hipotesis 2: En que año hay mas ataques  

In [23]:
df['Year'].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017.,   nan, 2016., 2015., 2014., 2013., 2012., 2011., 2010.,
       2009., 2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001.,
       2000., 1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993.,
       1992., 1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985.,
       1983., 1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975.,
       1974., 1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965.,
       1964., 1963., 1962., 1961., 1960., 1959., 1958., 1957., 1956.,
       1955., 1954., 1953., 1952., 1951., 1950., 1949., 1948., 1848.,
       1947., 1946., 1945., 1944., 1943., 1942., 1941., 1940., 1939.,
       1938., 1937., 1936., 1935., 1934., 1933., 1932., 1931., 1930.,
       1929., 1928., 1927., 1926., 1925., 1924., 1923., 1922., 1921.,
       1920., 1919., 1918., 1917., 1916., 1915., 1914., 1913., 1912.,
       1911., 1910., 1909., 1908., 1907., 1906., 1905., 1904., 1903.,
       1902., 1901.,

In [24]:
#Haciendo analisis de los datos de Year y quedandonos con los años mayor a 1000
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df.loc[df['Year'] <= 1000, 'Year'] = np.nan

print("NaN:", df['Year'].isna().sum())
print("Filas antes:", df.shape[0])


NaN: 126
Filas antes: 6485


In [25]:
#Eliminando los valor "nan"
df = df.dropna(subset=['Year'])
print("Filas después:", df.shape[0])

Filas después: 6359


In [26]:
moda_year = df['Year'].mode()[0]
moda_year

2015.0

### Columna Fatal y/n

In [27]:
df['Fatal Y/N'].unique()

array(['N', 'Y', 'F', 'M', nan, 'n', 'Nq', 'UNKNOWN', 2017, 'Y x 2', ' N',
       'y'], dtype=object)

In [28]:
#Quitamos espacios en blanco y ponemos todos en mayuscula, para unificar valores
df['Fatal Y/N'] = df['Fatal Y/N'].str.strip().str.upper()
df['Fatal Y/N'].unique()

array(['N', 'Y', 'F', 'M', nan, 'NQ', 'UNKNOWN', 'Y X 2'], dtype=object)

In [29]:
#Englobamos valores raros a 'Unkown'
df['Fatal Y/N'] = df['Fatal Y/N'].replace({
    'Y X 2': 'Y',
    'NQ': 'N'          
})

df['Fatal Y/N'].unique()

array(['N', 'Y', 'F', 'M', nan, 'UNKNOWN'], dtype=object)

In [30]:
#Convirtiendo "UNKNOWN", "F", "M", en nan
# todo lo que NO sea Y o N pasalo a np.nan
df.loc[~df['Fatal Y/N'].isin(['Y', 'N']), 'Fatal Y/N'] = np.nan

df['Fatal Y/N'].value_counts(dropna=False)

Fatal Y/N
N      4511
Y      1323
NaN     525
Name: count, dtype: int64

In [31]:
#eliminamos los nan 
df = df.dropna(subset=['Fatal Y/N'])

In [32]:
#Visualizando cambios y utilizando describe para observar cuales son los mas frequentes 
print(df['Fatal Y/N'].unique())

df['Fatal Y/N'].describe()

['N' 'Y']


count     5834
unique       2
top          N
freq      4511
Name: Fatal Y/N, dtype: object

In [33]:
#Observando el porcentaje con respecto a la muestra final con la que nos quedamos 
(df['Fatal Y/N'].value_counts(normalize=True)['N']) * 100

77.32259170380527

### Activity

In [34]:
#Viendo los valores que contiene Activity y sus cantidades
df['Activity'].value_counts()

Activity
Surfing                         1073
Swimming                         881
Fishing                          371
Spearfishing                     347
Wading                           163
                                ... 
Resting on body board              1
Diving in tuna net                 1
Free diving / spearfishing,        1
Commercial spearfishing            1
Swmming                            1
Name: count, Length: 1319, dtype: int64

In [35]:
#revisando cuantos nulos hay en columna Activity 
df['Activity'].isna().sum()

302

In [36]:
#Los pasamos a minusculas y quitamos espacios para ver si hay coincidencias entre los valores
df['Activity'] = df['Activity'].str.strip().str.lower()

In [37]:
#ver si tenia valores nulos escritos como print y pasarlos a valor np.nan
df[df['Activity'].astype(str).str.lower().isin(['nan', 'none', 'null', ''])]['Activity'].value_counts()

Activity
    1
Name: count, dtype: int64

In [38]:
#Lista de categorías frecuentes para poder agrupar valores
frequent_categories = [
    'surfing', 'swimming', 'fishing', 'spearfishing', 'wading', 'bathing',
    'diving', 'snorkeling', 'standing', 'scuba diving', 'body boarding',
    'boogie boarding', 'body surfing', 'kayaking', 'free diving', 'treading water',
    'fell overboard', 'pearl diving', 'surf skiing', 'windsurfing', 'floating',
    'walking', 'canoeing', 'kayak fishing', 'shark fishing', 'drifting', 'boating', 'sailing'
]

In [39]:
#funcion creada para que itere en la lista de frenquent_categories y si consigue alguna palabra que este en frequent_categories
#la almacene en la categoria correspondiente, sino que la incluya en un categoria llamada other
def map_activity(activity):
    if pd.isna(activity):
        return np.nan
    for category in frequent_categories:
        if category in activity.lower():
            return category
    return 'other'


In [40]:
#aplicamos la funcion map_activy en nuestra columna y vemos el filtrado 
df['Activity'] = df['Activity'].apply(map_activity)

# Ver resultados
print(df['Activity'].value_counts(dropna=False))

Activity
surfing            1235
swimming           1107
fishing            1024
other               717
diving              475
NaN                 302
wading              178
bathing             167
standing            139
snorkeling          130
body boarding        69
boogie boarding      55
fell overboard       51
floating             46
kayaking             40
treading water       36
surf skiing          22
walking              20
canoeing             12
sailing               9
Name: count, dtype: int64


### Columna "Country"

In [41]:
df_country = df[['Country']]

In [42]:
#Observacion de valor unicos
df_country['Country'].unique()

array(['Australia', 'US Virgin Islands', 'New Caledonia', 'USA',
       'French Polynesia', 'Samoa', 'Columbia', 'Costa Rica', 'Bahamas',
       'Puerto Rico', 'Spain', 'Canary Islands', 'South Africa',
       'Vanuatu', 'Jamaica', 'Israel', 'Mexico', 'Maldives',
       'Philippines', 'Turks and Caicos', 'Mozambique', 'Egypt',
       'Thailand', 'New Zealand', 'Hawaii', 'Honduras', 'Indonesia',
       'Morocco', 'Belize', 'Maldive Islands', 'Tobago', 'AUSTRALIA',
       'INDIA', 'TRINIDAD', 'BAHAMAS', 'SOUTH AFRICA', 'MEXICO',
       'NEW ZEALAND', 'EGYPT', 'BELIZE', 'Coral Sea', 'SPAIN', 'PORTUGAL',
       'SAMOA', 'COLOMBIA', 'ECUADOR', 'FRENCH POLYNESIA',
       'NEW CALEDONIA', 'TURKS and CaICOS', 'CUBA', 'BRAZIL', 'FIJI',
       'MeXICO', 'ENGLAND', 'JAPAN', 'INDONESIA', 'JAMAICA', 'MALDIVES',
       'THAILAND', 'COLUMBIA', 'British Overseas Territory', 'CANADA',
       'JORDAN', 'ST KITTS / NEVIS', 'ST MARTIN', 'SEYCHELLES',
       'PAPUA NEW GUINEA', 'ISRAEL', 'REUNION ISLAND', 

In [43]:
#unificando para que todo sea mayuscula y quitando espacio en blanco, se agrupan algunas.
df_country['Country'] = df_country['Country'].str.upper().str.strip()

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_13410/26686830.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_country['Country'] = df_country['Country'].str.upper().str.strip()


In [44]:
#Observamos que se unifican algunos valores 
df_country['Country'].value_counts()

Country
USA                               2267
AUSTRALIA                         1238
SOUTH AFRICA                       468
BAHAMAS                            127
NEW ZEALAND                        120
                                  ... 
FEDERATED STATES OF MICRONESIA       1
TRINIDAD                             1
RED SEA                              1
CORAL SEA                            1
BETWEEN PORTUGAL & INDIA             1
Name: count, Length: 191, dtype: int64

In [45]:
#lo pasamos a series para observar con cuantos valores unicos estamos tratando 
pd.Series(df_country['Country'].unique())

0                     AUSTRALIA
1             US VIRGIN ISLANDS
2                 NEW CALEDONIA
3                           USA
4              FRENCH POLYNESIA
                 ...           
187                  TASMAN SEA
188                       GHANA
189                        JAVA
190           MEDITERRANEAN SEA
191    BETWEEN PORTUGAL & INDIA
Length: 192, dtype: object

In [46]:
#Aqui toma el primer valor al separa valor/valor
df_country['Country'] = df_country['Country'].str.split('/').str[0].str.strip()

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_13410/1120493894.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_country['Country'] = df_country['Country'].str.split('/').str[0].str.strip()


In [47]:
other= {
    'ST KITTS / NEVIS': 'SAINT KITTS AND NEVIS',
    'ST KITTS': 'SAINT KITTS AND NEVIS',
    'NEVIS': 'SAINT KITTS AND NEVIS',
    'TRINIDAD & TOBAGO': 'TRINIDAD AND TOBAGO',
    'TRINIDAD': 'TRINIDAD AND TOBAGO',
    'TURKS & CAICOS': 'TURKS AND CAICOS',
    'ST. MARTIN': 'SAINT MARTIN',
    'ST. MAARTIN': 'SAINT MARTIN',
    'MALDIVE ISLANDS': 'MALDIVES',
    'COLUMBIA': 'COLOMBIA',

    'BRITISH VIRGIN ISLANDS': 'UNITED KINGDOM',
    'ST HELENA, BRITISH OVERSEAS TERRITORY': 'UNITED KINGDOM',
    'UNITED ARAB EMIRATES (UAE)': 'UNITED ARAB EMIRATES',
    'HAWAII': 'USA',
    'CANARY ISLANDS': 'SPAIN',
    'MALAYSIA': 'MALAYSIA',
    'CEYLON (SRI LANKA)': 'SRI LANKA',
    'EQUATORIAL GUINEA / CAMEROON': 'EQUATORIAL GUINEA',  
    'BETWEEN PORTUGAL & INDIA': 'PORTUGAL', 
    'CORAL SEA': 'OTHER',
    'ATLANTIC OCEAN': 'OTHER',
    'CARIBBEAN SEA': 'OTHER',
    'COAST OF AFRICA': 'OTHER',
    'RED SEA?': 'OTHER',
    'ASIA?': 'OTHER',
    'DIEGO GARCIA': 'OTHER', 
    'OKINAWA': 'JAPAN',
    'COLUMBIA': 'COLOMBIA',
    'TRINIDAD': 'TRINIDAD AND TOBAGO',
    'MALDIVE ISLANDS': 'MALDIVES',
    'TURKS': 'TURKS AND CAICOS',
    'REUNION ISLAND': 'OTHER',  

    'CORAL SEA': 'OTHER',
    'ATLANTIC OCEAN': 'OTHER',
    'GULF OF ADEN': 'OTHER',
    'TASMAN SEA': 'OTHER',
    'NORTH ATLANTIC OCEAN': 'OTHER',
    'SOUTH CHINA SEA': 'OTHER',
    'INDEPENDENT STATES': 'OTHER',  

    'RED SEA?' : 'OTHER',
    'ASIA?' : 'OTHER',
    'CEYLON (SRI LANKA)' : 'OTHER',


    # Entradas duplicadas u obsoletas
    'TOBAGO': 'TRINIDAD AND TOBAGO', 
    'BRITISH OVERSEAS TERRITORY': 'OTHER',  
    'PALESTINIAN TERRITORIES': 'OTHER',  
    'ANDAMAN ISLANDS': 'INDIA',  
    'EQUATORIAL GUINEA / CAMEROON': 'EQUATORIAL GUINEA',
    'NEW CALEDONIA': 'FRENCH POLYNESIA',   
    'CEYLON': 'SRI LANKA',
    'NORTHERN ARABIAN SEA': 'OTHER',
    'INDIAN OCEAN?': 'OTHER',
    'AFRICA': 'OTHER',

    'WESTERN SAMOA': 'SAMOA',
    'PACIFIC OCEAN': 'OTHER',
    'BRITISH ISLES': 'UNITED KINGDOM', 
    'NEW BRITAIN': 'PAPUA NEW GUINEA', 
    'JOHNSTON ISLAND': 'OTHER',
    'SOUTH PACIFIC OCEAN': 'OTHER',
    'WEST INDIES': 'OTHER', 
    'BURMA': 'MYANMAR',
    'BRITISH NEW GUINEA': 'PAPUA NEW GUINEA',
    'OCEAN': 'OTHER',
    'MEDITERRANEAN SEA': 'OTHER',
    'ROATAN': 'OTHER',
    'SOUTH CHINA SEA': 'OTHER',
    'KOREA': 'OTHER',
    'MID-PACIFC OCEAN': 'OTHER'
}

In [48]:
#reemplazamos valores que hacemos 
df_country['Country'] = df_country['Country'].replace(other)

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_13410/506533581.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_country['Country'] = df_country['Country'].replace(other)


In [49]:
df_country['Country'].value_counts()

Country
USA                     2268
AUSTRALIA               1238
SOUTH AFRICA             468
BAHAMAS                  127
NEW ZEALAND              120
                        ... 
SUDAN?                     1
ARGENTINA                  1
NETHERLANDS ANTILLES       1
SAUDI ARABIA               1
JAVA                       1
Name: count, Length: 148, dtype: int64

### Columna Date

In [50]:
df_date = df[['Date']]

In [51]:
df_date['Date'].unique()


array(['10th January', '8th January', '3rd January ', ..., 'Ca. 1543',
       'Ca 1588.04.00', 'Ca 1200-1500 A.D.'], dtype=object)

In [52]:
def false_words(s):
    eliminate_words = [
        'Before', 'After', 'No date', 'Late',
        'Circa', 'Ca.', 'Letter dated'
    ]
    
    s = str(s).lower()
    #Recorre la lista eliminate_words comprueba si alguna de esas palabras esta dentro del texto 
    #devuelve True si al menos encuenta una 
    for word in eliminate_words:
        if word.lower() in s:
            return True
    
    return False

In [53]:
def extract_full_month(s):
    #lista de los meses
    months_full = [
        'January', 'February', 'March', 'April', 'May', 'June',
        'July', 'August', 'September', 'October', 'November', 'December'
    ]
    
    #
    months_abbreviate = []
    for m in months_full:
        months_abbreviate.append(m[:3])
    #flags=re.I -> hace la bisque a mayuscula y minusculas
    #crea pares con sus el nombre completo del mes y su abreviatura ('January', 'Jan')
    for m_full, m_abbr in zip(months_full, months_abbreviate):
        #primera condicion busca el nombre completo del mes y la segunda la abreviatura
        if re.search(rf'\b{m_full}\b', s, flags=re.I) or \
           re.search(rf'\b{m_abbr}\b', s, flags=re.I):
            return m_full
    
    return None

In [54]:
#esta funcion trabaja con un solo valor
#su trabajo es limpiar, validar y extraer el mes 
def parse_month(x):
    if pd.isna(x):
        return None
        
    s = str(x).strip()
        
    #Descartar fechas inciertas
    if false_words(s):
        return None
        
    #Extraer mes
    return extract_full_month(s)

In [55]:
#Creamos una columna nueva para aplicar los cambios 
df_date['Month_full_strict'] = df_date['Date'].apply(parse_month)

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_13410/945431150.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_date['Month_full_strict'] = df_date['Date'].apply(parse_month)


In [56]:
#imprimimos uno al lado del otro para observar los cambios de las funciones creadas para quedarnos solo con meses
print(df_date[['Date', 'Month_full_strict']])

                          Date Month_full_strict
0                 10th January           January
1                  8th January           January
2                 3rd January            January
3                21st December          December
4                12th December          December
...                        ...               ...
6925                      1642              None
6927  Letter dated 10-Jan-1580              None
6930                  Ca. 1543              None
6931             Ca 1588.04.00              None
6932         Ca 1200-1500 A.D.              None

[5834 rows x 2 columns]


In [57]:
#hacemos el cambio en la columna date
df_date['Date'] = df_date['Date'].apply(parse_month)

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_13410/1468775711.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_date['Date'] = df_date['Date'].apply(parse_month)


In [58]:
#con dropna=False podemos ver cuantos None tenemos y que hay que eliminar, tenemos None 395
df_date['Date'].value_counts(dropna=False)

Date
July         672
August       554
September    532
June         471
January      451
October      439
April        411
December     408
None         395
May          386
March        386
November     384
February     345
Name: count, dtype: int64

In [59]:
#.notna() es un metodo que sirve para detectar los valores que no son nulos 
#el valor existe devuelve True, sino False
df_date = df_date[df_date['Date'].notna()]

In [60]:
#Revisando que eliminamos los None
df_date['Date'].value_counts(dropna=False)

Date
July         672
August       554
September    532
June         471
January      451
October      439
April        411
December     408
May          386
March        386
November     384
February     345
Name: count, dtype: int64

In [65]:
### hipotesis5 En que epoca del año ocurren mas ataques?


In [66]:
df_date['Date'].unique()

array(['January', 'December', 'November', 'October', 'September',
       'August', 'July', 'June', 'May', 'February', 'March', 'April'],
      dtype=object)

In [67]:
# Primero asegurarnos de que sean strings y sin espacios
df_date['Date'] = df_date['Date'].astype(str).str.strip()

In [68]:
trimesters = {
    'January': 'T1 (January,February, March)', 'February': 'T1 (January,February, March)', 'March': 'T1 (January,February, March)',
    'April': 'T2 (April, May, June)', 'May': 'T2 (April, May, June)', 'June': 'T2 (April, May, June)',
    'July': 'T3 (July, August, September)', 'August': 'T3 (July, August, September)', 'September': 'T3 (July, August, September)',
    'October': 'T4 (October, November, December)', 'November': 'T4 (October, November, December)', 'December': 'T4 (October, November, December)'
}

#aplicamos mapping 
df_date['Date'] = df_date['Date'].map(trimesters)


In [69]:
# Ver qué valores no se pudieron mapear
print(df_date[df_date['Date'].isna()]['Date'].unique())

[]


In [70]:
df_date['Date'].value_counts()

Date
T3 (July, August, September)        1758
T2 (April, May, June)               1268
T4 (October, November, December)    1231
T1 (January,February, March)        1182
Name: count, dtype: int64